In [1]:
pip install numpy opencv-python scikit-image scikit-learn qiskit qiskit-machine-learning matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.6 MB/s eta 0:00:00


In [2]:
import os
import cv2
import numpy as np
from skimage.segmentation import chan_vese
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

# Qiskit for Quantum SVM Benchmark
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel

# ==========================================
# STEP 1 & 2: Preprocessing & Chan-Vese
# ==========================================
def preprocess_and_segment(img_path):
    # Load image in grayscale
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # 1. Denoise with Bilateral Filter
    denoised = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)

    # 2. Enhance contrast using CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised)

    # 3. Chan-Vese Active Contour Segmentation
    # Normalize image to [0, 1] for skimage
    img_norm = enhanced / 255.0
    cv_mask = chan_vese(img_norm, mu=0.25, lambda1=1, lambda2=1, tol=1e-3, max_num_iter=100)

    # Convert mask back to uint8 binary image [0, 255]
    segmented_img = (cv_mask * 255).astype(np.uint8)
    return segmented_img

# ==========================================
# STEP 3: SIFT Feature Extraction
# ==========================================
sift = cv2.SIFT_create()

def extract_sift_descriptors(image):
    keypoints, descriptors = sift.detectAndCompute(image, None) #
    return descriptors

def process_dataset(image_paths):
    segmented_images = []
    all_descriptors = []
    image_descriptors_map = []

    for path in image_paths:
        seg_img = preprocess_and_segment(path)
        if seg_img is not None:
            desc = extract_sift_descriptors(seg_img)
            segmented_images.append(seg_img)
            image_descriptors_map.append(desc)
            if desc is not None:
                all_descriptors.append(desc)

    return image_descriptors_map, all_descriptors

# ==========================================
# STEP 3.5: Bag of Visual Words (BoVW)
# ==========================================
def build_bovw_histograms(image_descriptors_map, kmeans_model, num_clusters):
    X_bovw = []
    for desc in image_descriptors_map:
        histogram = np.zeros(num_clusters)
        if desc is not None and len(desc) > 0:
            words = kmeans_model.predict(desc)
            for word in words:
                histogram[word] += 1
            # Normalize histogram
            histogram = histogram / np.sum(histogram) if np.sum(histogram) > 0 else histogram
        X_bovw.append(histogram)
    return np.array(X_bovw)

# ==========================================
# STEP 4: Execution Pipeline
# ==========================================
def train_chd_pipeline(normal_img_paths, chd_img_paths, num_clusters=50):
    # Setup labels: 0 for Normal, 1 for CHD
    all_paths = normal_img_paths + chd_img_paths
    y = np.array([0] * len(normal_img_paths) + [1] * len(chd_img_paths))

    print("Extracting features and segmenting images...")
    image_desc_map, all_descriptors = process_dataset(all_paths)

    # Stack all descriptors to fit K-Means
    stacked_descriptors = np.vstack([d for d in all_descriptors if d is not None])

    print(f"Clustering visual words into {num_clusters} clusters...")
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    kmeans.fit(stacked_descriptors)

    # Build feature vectors
    X = build_bovw_histograms(image_desc_map, kmeans, num_clusters)

    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --- 1. Classical SVM ---
    print("\nTraining Classical SVM...")
    svm_model = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
    svm_model.fit(X_train_scaled, y_train)

    y_pred_svm = svm_model.predict(X_test_scaled)
    print("--- Classical SVM Performance ---")
    print(classification_report(y_test, y_pred_svm, target_names=['Normal', 'CHD']))

    # --- 2. Quantum SVM Benchmark ---
    print("\nEvaluating Quantum SVM (QSVM)...")
    # Reduce feature dimension dynamically if needed for Quantum Feature Mapping
    num_qubits = min(X_train_scaled.shape[1], 8) # Using 8 qubits as benchmark setup
    feature_map = ZZFeatureMap(feature_dimension=num_qubits, reps=2)
    quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)

    qsvm_model = SVC(kernel=quantum_kernel.evaluate)
    qsvm_model.fit(X_train_scaled[:, :num_qubits], y_train)

    y_pred_qsvm = qsvm_model.predict(X_test_scaled[:, :num_qubits])
    print("--- QSVM Benchmark Performance ---")
    print(classification_report(y_test, y_pred_qsvm, target_names=['Normal', 'CHD']))

    return svm_model, qsvm_model, kmeans, scaler